# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [9]:
# imports
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents, fetch_website_links
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [3]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [2]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [17]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

system_prompt = """
You are a helpfull assistant for explain code and give example. Answers with Indonesian language.
"""

In [18]:
message = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

In [19]:
# Get gpt-4o-mini to answer, with streaming

openai = OpenAI()

stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=message,
    stream=True
)

response = ""
display_handle = display(Markdown(), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

Kode yang Anda berikan adalah sebuah ekspresi generator menggunakan `yield from`. Mari kita bahas bagian-bagian dari kode tersebut.

### Penjelasan Kode
1. **`{book.get("author") for book in books if book.get("author")}`**:
   - Ini adalah *set comprehension* yang digunakan untuk membuat sebuah set baru.
   - `book` di sini adalah elemen yang diambil dari `books`, yang diasumsikan adalah sebuah list (atau iterable) dari objek-objek (mungkin kamus).
   - `book.get("author")` mencoba untuk mendapatkan nilai dari kunci `"author"` pada objek `book`.
   - `if book.get("author")` adalah kondisi yang memastikan bahwa hanya buku yang memiliki `author` yang akan dimasukkan ke dalam set.
   - Dengan demikian, set ini akan berisi semua penulis (`author`) dari buku-buku yang ada dalam `books`, dan jika ada penulis yang sama, hanya akan muncul sekali karena karakteristik set adalah unik.

2. **`yield from ...`**:
   - `yield from` adalah cara untuk menghasilkan elemen-elemen dari sebuah iterable dalam sebuah generator.
   - Ini akan mengembalikan elemen-elemen dari set yang dihasilkan oleh *set comprehension* ke pemanggil fungsi, satu per satu.

### Apa yang Dihasilkan Kode Ini?
Kode ini akan menghasilkan semua penulis dari buku-buku yang terdapat dalam `books`, dan akan menghindari duplikasi penulis dengan menggunakan set. Jika fungsi ini dipanggil dalam konteks generator, maka akan mengembalikan penulis satu per satu.

### Contoh
Berikut adalah contoh yang lebih konkret:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},
    {"title": "Book 4", "author": None},
    {"title": "Book 5"}
]

def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Menggunakan generator
for author in get_authors(books):
    print(author)
```

### Output
```
Author A
Author B
```

Dalam contoh di atas, kita memiliki daftar `books`. Fungsi `get_authors` kemudian menggunakan kode yang Anda berikan untuk mengekstrak penulis. Hasilnya hanya akan mengembalikan "Author A" dan "Author B", tanpa duplikasi. Penulis yang tidak ada (None atau tidak didefinisikan) diabaikan.

In [20]:
# Get Llama 3.2 to answer

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

stream_llama = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=message,
    stream=True
)

response = ""
display_stream = display(Markdown(), display_id=True)

for chunk in stream_llama:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_stream.display_id)

Pesan ini menggunakan bahasa Python, di mana `yield` adalah keyword yang digunakan untuk membuat generator.

Code di atas melibatkan beberapa objek dan operator:

*   `{...}`: pengambilan elemen dari iterable.
*   `book.get("author")`: mengakses nilai "author" dari sebuah objek book. Method `get()` digunakan untuk mendapatkan nilai pada kunci tertentu pada dict, jika kunci tidak ada maka nilainya akan diambil sebagai None.
*   `for book in books if book.get("author")` : loopi berapa banyak book yang memiliki atribut "author".

Perlu diingat bahwa variabel `books` ditidak disebutkan dalam kode tersebut. Misalkan `books` adalah sebuah collection dari objek (dapat berupa array, list atau apa saja) yang dikembangkan di atas.

Jika kita mengganti code:

*   `{book.get("author") for book in books if book.get("author").get("author")}` : akan lebih kompleks dan memerlukan library seperti `pandas` pada library ini kita harus memastikan bahwa tidak ada kesalahan pada data (misalkan dengan mendapatkan nilai "author" dari value yang diambil hanya dengan cara `.get()`).
*   `{book.get("author") for book in books if 'author' in book}` : lebih fleksibel dan tidak terikat untuk menggunakan sebuah method `get()`.
*   `{book.get("author") for book in books if hasattr(book, "author")}` : sangat direkomendasikan karena menggunakan fitur Python seperti `hasattr()` atau `dict.get()` yang akan menghindari kesalahan jika nilai "author" adalah None.

Code di atas juga memiliki keterbatasan yaitu tidak dapat digunakan untuk mendapatkan struktur dari data books. Contohnya, jika kita mendaftar buku dalam bentuk berikut:

`[
{
    'title': "Buku 1",
    'author' : "Pengarang Pertama"
}, 
{
    'title' : "Buku 2",
}
]`

Hasil dari code di atas akan hanya menampilkan nilai "author" saja.